In [1]:
class Variable:
    
    def __init__(self, data):
        self.data = data

In [3]:
import numpy as np

x = np.array([[1, 2, 3], [4, 5, 6]])
x = Variable(x)
x.data

array([[1, 2, 3],
       [4, 5, 6]])

In [4]:
x.data.ndim

2

In [5]:
class Function:

    def __call__(self, input):
        x = input.data
        y = x ** 2 
        output = Variable(y)
        return output

In [6]:
f = Function()
y = f(x)
y.data

array([[ 1,  4,  9],
       [16, 25, 36]])

In [13]:
class Function:
    def __call__(self, input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        return output
    
    def forward(self, x):
        raise NotImplementedError()

In [14]:
class Square(Function):
    def forward(self, x):
        return x ** 2

In [16]:
f = Square()
y = f(x)
y.data

array([[ 1,  4,  9],
       [16, 25, 36]])

In [17]:
class Exp(Function):
    def forward(self, x):
        return np.exp(x)

In [18]:
def numerical_diff(f, x, eps=1e-4):
    x0 = Variable(x.data - eps)
    x1 = Variable(x.data + eps)
    y0 = f(x0)
    y1 = f(x1)
    return (y1.data - y0.data) / (2 * eps)

In [19]:
class Variable:

    def __init__(self, data):
        self.data = data
        self.grad = None

In [21]:
class Function:

    def __call__(self, input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        self.input = input
        return output
    
    def forward(self, x):
        raise NotImplementedError()
    
    def backward(self, gy):
        raise NotImplementedError()

In [22]:
class Square(Function):
    
    def forward(self, x):
        return x ** 2
    
    def backward(self, gy):
        x = self.input.data
        gx = 2 * x * gy
        return gx

In [23]:
class Exp(Function):
    
    def forward(self, x):
        return np.exp(x)
    
    def backward(self, gy):
        x = self.input.data
        gx = np.exp(x) * gy
        return gx

In [25]:
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(0.5))
a = A(x)
b = B(a)
y = C(b)

In [28]:
y.grad = np.array(1.0)
b.grad = C.backward(y.grad)
a.grad = B.backward(b.grad)
x.grad = A.backward(a.grad)
x.grad

np.float64(3.297442541400256)

In [36]:
class Variable:

    def __init__(self, data):
        self.data = data
        self.grad = None
        self.creator = None

    def set_creator(self, func):
        self.creator = func
    
class Function:

    def __call__(self, input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        output.set_creator(self)
        self.input = input
        self.output = output
        return output
    
    def forward(self, x):
        raise NotImplementedError()
    
    def backward(self, gy):
        raise NotImplementedError()
        

In [37]:
class Square(Function):
    
    def forward(self, x):
        return x ** 2
    
    def backward(self, gy):
        x = self.input.data
        gx = 2 * x * gy
        return gx

class Exp(Function):
    
    def forward(self, x):
        return np.exp(x)
    
    def backward(self, gy):
        x = self.input.data
        gx = np.exp(x) * gy
        return gx

In [40]:
A = Square()
B = Exp()
C = Square()

a = A(x)
b = B(a)
y = C(b)

assert y.creator == C
assert y.creator.input == b

In [42]:
y.grad = np.array(1.0)

C = y.creator
b = C.input
b.grad = C.backward(y.grad)
b.grad

np.float64(2.568050833375483)

In [ ]:
class Variable:

    def __init__(self, data):
        self.data = data
        self.grad = None
        self.creator = None
    
    def set_creator(self, func):
        self.creator = func

    def backward(self):
        f = self.creator
        if f is not None:
            x = f.input
            x.grad = f.backward(self.grad)
            x.backward()

In [44]:
x = Variable(np.array(10.0))
a = A(x)
b = B(a)
y = C(b)
y.grad = np.array(1.0)
y.backward()
x.grad

np.float64(2.8903895072503e+88)

In [45]:
class Variable:

    def __init__(self, data):
        self.data = data
        self.grad = None
        self.creator = None
    
    def set_creator(self, func):
        self.creator = func

    def backward(self):
        funcs = [self.creator]
        while funcs:
            f = funcs.pop()
            x, y = f.input, f.output
            x.grad = f.backward(y.grad)

            if x.creator is not None:
                funcs.append(x.creator)

In [49]:
def square(x):
    return Square()(x)

def exp(x):
    return Exp()(x)

In [51]:
x = Variable(np.array(0.5))
a = square(x)
b = exp(a)
y = square(b)

y.grad = np.array(1.0)
y.backward()
x.grad

np.float64(3.297442541400256)

In [57]:
class Variable:

    def __init__(self, data):
        if data is not None:
            if not isinstance(data, np.ndarray):
                raise TypeError("{} is not supported".format(type(data)))
        self.data = data
        self.grad = None
        self.creator = None

    def set_creator(self, func):
        self.creator = func
    
    def backward(self):
        if self.grad is None:
            self.grad = np.ones_like(self.data)
        funcs = [self.creator]
        while funcs:
            f = funcs.pop()
            x, y = f.input, f.output
            x.grad = f.backward(y.grad)

            if x.creator is not None:
                funcs.append(x.creator)

In [59]:
def as_array(x):
    if np.isscalar:
        return np.array(x)
    return x

In [60]:
class Function:
    def __call__(self, input):
        x = input.data
        y = self.forward(x)
        output = Variable(as_array(y))
        output.set_creator(self)
        self.input = input
        self.output = output
        return output 
    
    def forward(self, x):
        raise NotImplementedError()
    
    def backward(self, gy):
        raise NotImplementedError()